In [50]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_selection import SelectPercentile, chi2

from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report, accuracy_score, f1_score

In [36]:
data = pd.read_excel("../data/job_classification.ods", engine = "odf", dtype = str)

data.head()

,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


In [37]:
data = data.dropna(axis = 0)
data.shape

(8073, 6)

In [38]:
def filler_location(location):
    result = re.findall("\\,\\s[A-Z]{2}$", location)
    if len(result) > 0:
        return result[0][2:]
    else:
        return location


data["location"] = data["location"].apply(filler_location)

In [39]:
target = "career_level"
x = data.drop(target, axis = 1)
y = data[target]

In [55]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [41]:
transformers = ColumnTransformer(transformers = [
    ("title", TfidfVectorizer(stop_words="english"), "title"),
    ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
    ("description", TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df = 0.01, max_df = 0.95), "description"),
    ("function", OneHotEncoder(handle_unknown="ignore"), ["function"]),
    ("industry", TfidfVectorizer(stop_words="english"), "industry")
])

In [42]:
normal_model = Pipeline(steps=[
    ("transformer", transformers),
    ("feature_selector", SelectPercentile(chi2, percentile = 5)),
    ("classifier", LinearSVC())
])

In [43]:
normal_model.fit(x_train, y_train)
y_normal_predict = normal_model.predict(x_test)
print(classification_report(y_test, y_normal_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.53      0.36      0.43       192
         director_business_unit_leader       0.90      0.64      0.75        14
                   manager_team_leader       0.67      0.67      0.67       534
managing_director_small_medium_company       1.00      1.00      1.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       1.00      0.17      0.29         6

                              accuracy                           0.76      1615
                             macro avg       0.82      0.62      0.67      1615
                          weighted avg       0.75      0.76      0.75      1615



In [44]:
balanced_model = Pipeline(steps=[
    ("transformer", transformers),
    ("feature_selector", SelectPercentile(chi2, percentile = 5)),
    ("classifier", LinearSVC(class_weight = "balanced")),
])

In [45]:
balanced_model.fit(x_train, y_train)
y_balanced_predict = normal_model.predict(x_test)
print(classification_report(y_test, y_balanced_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.53      0.36      0.43       192
         director_business_unit_leader       0.90      0.64      0.75        14
                   manager_team_leader       0.67      0.67      0.67       534
managing_director_small_medium_company       1.00      1.00      1.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       1.00      0.17      0.29         6

                              accuracy                           0.76      1615
                             macro avg       0.82      0.62      0.67      1615
                          weighted avg       0.75      0.76      0.75      1615



In [56]:
over_sampling = RandomOverSampler(random_state=42, sampling_strategy = {
    "managing_director_small_medium_company" : 100,
    "specialist" : 100,
    "director_business_unit_leader" : 100,
    "bereichsleiter" : 1000
})

x_train, y_train = over_sampling.fit_resample(x_train, y_train)
print(y_train.value_counts())

career_level
senior_specialist_or_project_manager      3469
manager_team_leader                       2138
bereichsleiter                            1000
specialist                                 100
director_business_unit_leader              100
managing_director_small_medium_company     100
Name: count, dtype: int64


In [57]:
oversampling_model = Pipeline(
    steps=[
        ("transformers", transformers),
        ("feature_selector", SelectPercentile(chi2, percentile = 5)),
        ("classifier", LinearSVC())
    ]
)

In [58]:
oversampling_model.fit(x_train, y_train)

y_over_predict = oversampling_model.predict(x_test)

print(classification_report(y_test, y_over_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.50      0.43      0.46       192
         director_business_unit_leader       0.73      0.57      0.64        14
                   manager_team_leader       0.67      0.58      0.62       534
managing_director_small_medium_company       1.00      1.00      1.00         1
  senior_specialist_or_project_manager       0.81      0.91      0.85       868
                            specialist       1.00      0.17      0.29         6

                              accuracy                           0.74      1615
                             macro avg       0.78      0.61      0.64      1615
                          weighted avg       0.73      0.74      0.73      1615



In [59]:
results = pd.DataFrame({
    "Experiment": [
        "Normal",
        "Class Weight Balanced",
        "SMOTEN"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_normal_predict),
        accuracy_score(y_test, y_balanced_predict),
        accuracy_score(y_test, y_over_predict)
    ],
    "Macro F1": [
        f1_score(y_test, y_normal_predict, average="macro"),
        f1_score(y_test, y_balanced_predict, average="macro"),
        f1_score(y_test, y_over_predict, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_test, y_normal_predict, average="weighted"),
        f1_score(y_test, y_balanced_predict, average="weighted"),
        f1_score(y_test, y_over_predict, average="weighted")
    ]
})

results

,Experiment,Accuracy,Macro F1,Weighted F1
0,Normal,0.759133,0.668383,0.749494
1,Class Weight Balanced,0.759133,0.668383,0.749494
2,SMOTEN,0.736842,0.644388,0.727663
